# Screening ARR 2019 Temporal Patterns for Embedded Burst Errors

Companion notebook to the published article. Screens a temporal pattern's rainfall increments for a suspected embedded-burst extraction artefact (two large bursts ~24h apart).

**Not yet done:** programmatic retrieval from the ARR Data Hub API -- see the TODO cell near the end. The screening function itself is complete and unit-tested.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## The screening function

In [1]:
def screen_embedded_bursts(increments, timestep_hours, flag_window_hours=24, tolerance_hours=1.5):
    """Flag a temporal pattern for a suspected embedded-burst error.

    Finds the two largest rainfall increments and checks whether they sit
    suspiciously close to `flag_window_hours` apart -- the signature of a
    duplication/extraction artefact rather than a genuine double-peaked storm.
    """
    increments = np.asarray(increments, dtype=float)
    order = np.argsort(increments)[::-1]
    idx1, idx2 = order[0], order[1]
    time1, time2 = idx1 * timestep_hours, idx2 * timestep_hours
    separation = abs(time1 - time2)
    flagged = abs(separation - flag_window_hours) <= tolerance_hours
    return {
        "flagged": bool(flagged),
        "largest_increment": float(increments[idx1]),
        "largest_time_hr": float(time1),
        "second_increment": float(increments[idx2]),
        "second_time_hr": float(time2),
        "separation_hr": float(separation),
    }

## Validation

In [2]:
rng = np.random.default_rng(0)

clean = np.abs(rng.normal(2, 1, 48))
clean[20] = 25.0
assert screen_embedded_bursts(clean, timestep_hours=1)["flagged"] is False

suspect = np.abs(rng.normal(2, 1, 48))
suspect[10] = 25.0
suspect[34] = 24.5
assert screen_embedded_bursts(suspect, timestep_hours=1)["flagged"] is True

print("Both test cases pass.")

Both test cases pass.


## Visualise clean vs. flagged

In [3]:
t = np.arange(48)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
axes[0].bar(t, clean, color="#2e8bc0", width=0.8)
axes[0].set_title("Clean pattern — not flagged")
axes[0].set_xlabel("Hour"); axes[0].set_ylabel("Rainfall increment (mm)")
axes[1].bar(t, suspect, color="#c0392b", width=0.8)
axes[1].set_title("Suspect pattern — flagged")
axes[1].set_xlabel("Hour")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

<Figure size 1100x500 with 2 Axes>

## TODO -- ARR Data Hub retrieval

Not completed: pulling temporal patterns programmatically from the ARR Data Hub. Requires `requests` and Data Hub API credentials/endpoint details Lindsay has access to.

Suggested shape once the retrieval is sorted:

```python
def fetch_arr_temporal_patterns(duration_hours, region):
    # TODO: call the ARR Data Hub API, return a dict of
    #   {pattern_id: increments_array}
    # for all 10 patterns for this duration/region.
    raise NotImplementedError

patterns = fetch_arr_temporal_patterns(duration_hours=24, region='...')
for pattern_id, increments in patterns.items():
    result = screen_embedded_bursts(increments, timestep_hours=1)
    flag = ' <-- FLAGGED' if result['flagged'] else ''
    print(f"{pattern_id}: separation={result['separation_hr']:.1f}h{flag}")
```

## Reference

Ladson, A.R. (2021). Review of temporal patterns from Australian Rainfall and Runoff 2019. *39th Hydrology and Water Resources Symposium.*